# Cenário 4: Erros com Média Não-Zero - Simulação MRLS

## Objetivo
Estudar o comportamento do estimador MQO quando os erros têm média não nula (viés sistemático nos erros).

## Especificação do Modelo
- $Y_i = eta_0 + eta_1 X_i + psilon_i$
- Aqui: $psilon_i = u + u_i$, com $u_i im N(0,igma^2)$ e $u 
eq 0$ (média não-zero).

## Estrutura do Estudo
Seguir o mesmo algoritmo e etapas dos notebooks anteriores: imports → parâmetros → funções utilitárias → gerar X uma vez → B repetições → ajustar OLS → agregados, diagnósticos e salvar.
---

In [ ]:
# 1. Importar bibliotecas
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
# 2. Parâmetros do cenário
BETA0_TRUE = 1.0
BETA1_TRUE = 2.0
SIGMA = 1.0

# Parâmetro de média não-zero dos erros (viés nos erros)
MU_ERROR = 1.0  # média dos erros: ε = MU_ERROR + Normal(0, SIGMA^2)

SAMPLE_SIZES = [30, 100]
NUM_REPLICATIONS = 1000
ALPHA = 0.05
SEED = 44
X_LOW = 0.0
X_HIGH = 10.0

print('Parâmetros do Cenário 4: Erro com média não-zero')
print(f'  μ (erro) = {MU_ERROR}')
print(f'  σ = {SIGMA}')
print(f'  n = {SAMPLE_SIZES}')
print(f'  B = {NUM_REPLICATIONS}')

In [ ]:
# 3. Funções utilitárias (mesmas usadas nos cenários anteriores)
def generate_X(n, rng, x_low, x_high):
    return rng.uniform(x_low, x_high, size=n)

def generate_Y_scenario4_constant(X, beta0, beta1, sigma, mu_error, rng):
    # Erros com média não-zero constante: epsilon = mu_error + u, u ~ N(0,sigma^2)
    n = len(X)
    u = rng.normal(0, sigma, size=n)
    epsilon = mu_error + u
    Y = beta0 + beta1 * X + epsilon
    return Y, epsilon

def fit_ols_model(X, Y, alpha):
    X_with_const = sm.add_constant(X, has_constant='add')
    model = sm.OLS(Y, X_with_const).fit()
    ci = model.conf_int(alpha=alpha)
    return {
        'beta0_hat': float(model.params[0]),
        'beta1_hat': float(model.params[1]),
        'se_beta0': float(model.bse[0]),
        'se_beta1': float(model.bse[1]),
        'ci_beta0_low': float(ci[0, 0]),
        'ci_beta0_high': float(ci[0, 1]),
        'ci_beta1_low': float(ci[1, 0]),
        'ci_beta1_high': float(ci[1, 1]),
        't_stat_beta1': float(model.tvalues[1]),
        'pvalue_beta1': float(model.pvalues[1]),
        'resid': np.array(model.resid),
        'fitted': np.array(model.fittedvalues),
        'r_squared': float(model.rsquared),
    }

def summarize_estimator(estimates, true_value):
    mean_est = np.mean(estimates)
    bias = mean_est - true_value
    variance = np.var(estimates, ddof=1)
    mse = np.mean((estimates - true_value) ** 2)
    return {'mean': mean_est, 'bias': bias, 'variance': variance, 'mse': mse, 'std_dev': np.sqrt(variance)}

def compute_coverage(ci_low, ci_high, true_value):
    coverage = np.mean((ci_low <= true_value) & (true_value <= ci_high))
    return float(coverage)

def compute_rejection_rate(pvalues, alpha):
    return float(np.mean(pvalues < alpha))

In [ ]:
# 4. Preparar simulação: gerar X uma única vez por n e executar B repetições
rng = np.random.default_rng(SEED)
results = {}

for n in SAMPLE_SIZES:
    X = generate_X(n, rng, X_LOW, X_HIGH)
    beta0_hats = []
    beta1_hats = []
    se_beta0_list = []
    se_beta1_list = []
    ci_beta0_low_list = []
    ci_beta0_high_list = []
    ci_beta1_low_list = []
    ci_beta1_high_list = []
    pvalue_beta1_list = []
    residuals_diagnostic = None
    fitted_diagnostic = None
    r_squared_list = []

    print(f"
{'='*60}")
    print(f"Simulação para n = {n}")
    print(f"{'='*60}")

    for rep in tqdm(range(NUM_REPLICATIONS), desc=f"Repetições (n={n})"):
        Y, epsilon = generate_Y_scenario4_constant(X, BETA0_TRUE, BETA1_TRUE, SIGMA, MU_ERROR, rng)
        fit_results = fit_ols_model(X, Y, ALPHA)
        beta0_hats.append(fit_results['beta0_hat'])
        beta1_hats.append(fit_results['beta1_hat'])
        se_beta0_list.append(fit_results['se_beta0'])
        se_beta1_list.append(fit_results['se_beta1'])
        ci_beta0_low_list.append(fit_results['ci_beta0_low'])
        ci_beta0_high_list.append(fit_results['ci_beta0_high'])
        ci_beta1_low_list.append(fit_results['ci_beta1_low'])
        ci_beta1_high_list.append(fit_results['ci_beta1_high'])
        pvalue_beta1_list.append(fit_results['pvalue_beta1'])
        if rep == 0:
            residuals_diagnostic = fit_results['resid']
            fitted_diagnostic = fit_results['fitted']
        r_squared_list.append(fit_results['r_squared'])

    results[n] = {
        'X': X,
        'beta0_hats': np.array(beta0_hats),
        'beta1_hats': np.array(beta1_hats),
        'se_beta0': np.array(se_beta0_list),
        'se_beta1': np.array(se_beta1_list),
        'ci_beta0_low': np.array(ci_beta0_low_list),
        'ci_beta0_high': np.array(ci_beta0_high_list),
        'ci_beta1_low': np.array(ci_beta1_low_list),
        'ci_beta1_high': np.array(ci_beta1_high_list),
        'pvalue_beta1': np.array(pvalue_beta1_list),
        'residuals_diagnostic': residuals_diagnostic,
        'fitted_diagnostic': fitted_diagnostic,
        'r_squared': np.array(r_squared_list),
    }
    print(f"✓ Simulação concluída para n = {n}")

print('
' + '='*60)
print('Todas as simulações concluídas para Cenário 4')
print('='*60)

In [ ]:
# 5. Estimação: resumir propriedades dos estimadores
estimation_summary = []
for n in SAMPLE_SIZES:
    s0 = summarize_estimator(results[n]['beta0_hats'], BETA0_TRUE)
    estimation_summary.append({'n': n, 'parâmetro': 'β₀', 'valor_verdadeiro': BETA0_TRUE, 'média_estimador': s0['mean'], 'viés': s0['bias'], 'variância': s0['variance'], 'EQM': s0['mse']})
    s1 = summarize_estimator(results[n]['beta1_hats'], BETA1_TRUE)
    estimation_summary.append({'n': n, 'parâmetro': 'β₁', 'valor_verdadeiro': BETA1_TRUE, 'média_estimador': s1['mean'], 'viés': s1['bias'], 'variância': s1['variance'], 'EQM': s1['mse']})
estimation_df = pd.DataFrame(estimation_summary)
print('
' + '='*100)
print('TABELA: ESTIMAÇÃO (Cenário 4)')
print('='*100)
print(estimation_df.to_string(index=False))

all_results = {'estimation': estimation_df}

In [ ]:
# 6. Inferência: cobertura empírica dos ICs
inference_summary = []
for n in SAMPLE_SIZES:
    cov_b0 = compute_coverage(results[n]['ci_beta0_low'], results[n]['ci_beta0_high'], BETA0_TRUE)
    inference_summary.append({'n': n, 'parâmetro': 'β₀', 'cobertura_empírica': cov_b0, 'cobertura_teórica': 1-ALPHA, 'diferença': cov_b0 - (1-ALPHA)})
    cov_b1 = compute_coverage(results[n]['ci_beta1_low'], results[n]['ci_beta1_high'], BETA1_TRUE)
    inference_summary.append({'n': n, 'parâmetro': 'β₁', 'cobertura_empírica': cov_b1, 'cobertura_teórica': 1-ALPHA, 'diferença': cov_b1 - (1-ALPHA)})
inference_df = pd.DataFrame(inference_summary)
print('
' + '='*100)
print('TABELA: COBERTURA EMPÍRICA (Cenário 4)')
print('='*100)
print(inference_df.to_string(index=False))
all_results['inference'] = inference_df

In [ ]:
# 7. Testes: taxa de rejeição / poder
tests_summary = []
for n in SAMPLE_SIZES:
    power = compute_rejection_rate(results[n]['pvalue_beta1'], ALPHA)
    tests_summary.append({'n': n, 'hipótese': 'H1: β1 ≠ 0', 'β1_verdadeiro': BETA1_TRUE, 'taxa_rejeição (poder)': power})
tests_df = pd.DataFrame(tests_summary)
print('
' + '='*100)
print('TABELA: TESTES (Cenário 4)')
print('='*100)
print(tests_df.to_string(index=False))
all_results['tests'] = tests_df

In [ ]:
# 8. Diagnóstico e gráficos (salvar figuras)
fig, axes = plt.subplots(len(SAMPLE_SIZES), 3, figsize=(15, 5*len(SAMPLE_SIZES)))
if len(SAMPLE_SIZES) == 1:
    axes = axes.reshape(1, -1)
for idx, n in enumerate(SAMPLE_SIZES):
    beta1_samples = results[n]['beta1_hats']
    ax = axes[idx, 0]
    ax.hist(beta1_samples, bins=40, density=True, alpha=0.7, color='steelblue')
    mu_b = np.mean(beta1_samples); sd_b = np.std(beta1_samples)
    x_range = np.linspace(mu_b - 4*sd_b, mu_b + 4*sd_b, 100)
    ax.plot(x_range, stats.norm.pdf(x_range, mu_b, sd_b), 'r-', linewidth=2)
    ax.axvline(BETA1_TRUE, color='green', linestyle='--')
    ax.set_title(f'Histograma β̂₁ (n={n})')
    ax = axes[idx, 1]
    ax.boxplot([beta1_samples], labels=[f'n={n}'])
    ax.axhline(BETA1_TRUE, color='green', linestyle='--')
    ax.set_title(f'Boxplot β̂₁ (n={n})')
    ax = axes[idx, 2]
    stats.probplot(beta1_samples, dist='norm', plot=ax)
    ax.set_title(f'QQ-plot β̂₁ (n={n})')
plt.tight_layout()
plt.savefig('cenario4_diagnostico_beta1.png', dpi=100, bbox_inches='tight')
plt.show()

# Resíduos diagnostics (maior n)
n_diag = max(SAMPLE_SIZES)
residuals = results[n_diag]['residuals_diagnostic']
fitted = results[n_diag]['fitted_diagnostic']
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].scatter(fitted, residuals, alpha=0.6, s=40)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Resíduos vs Ajustados')
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('QQ-plot dos resíduos')
plt.tight_layout()
plt.savefig('cenario4_diagnostico_residuos.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# 9. Salvar resultados em CSV
output_dir = 'cenario4_resultados'
os.makedirs(output_dir, exist_ok=True)
for name, df in all_results.items():
    filepath = os.path.join(output_dir, f'cenario4_{name}.csv')
    df.to_csv(filepath, index=False)
    print(f'✓ Salvo: {filepath}')
print(f'✓ Arquivos salvos em: {output_dir}/')

# Cenário 4: Erros com Média Não-Zero - Simulação MRLS

## Objetivo
Estudar o comportamento do estimador MQO quando os erros têm média diferente de zero (viola a hipótese E[ε|X]=0).

## Especificação do Cenário
- $Y_i = eta_0 + eta_1 X_i + psilon_i$
- Aqui consideramos $psilon_i = u + u_i$, onde $u_i im athcal{N}(0,igma^2)$ e $u 
eq 0$ (média deslocada).
- Também incluímos uma variante opcional com média dependente de X: $psilon_i = g(X_i) + u_i$ (ex.: $g(X)=amma X$).

---

## Algoritmo e Estrutura: mesma sistemática dos cenários anteriores

In [ ]:
# 1. Importar bibliotecas
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

In [ ]:
# 2. Parâmetros do cenário
BETA0_TRUE = 1.0
BETA1_TRUE = 2.0
SIGMA = 1.0

# Média deslocada dos erros (constante)
MU_ERROR = 1.0
# Variante: média dependente de X (defina GAMMA=0 para desativar)
GAMMA = 0.0  # se >0, usa epsilon = GAMMA * X + u

SAMPLE_SIZES = [30, 100]
NUM_REPLICATIONS = 1000
ALPHA = 0.05
SEED = 44
X_LOW = 0.0
X_HIGH = 10.0

print('Parâmetros do Cenário 4: Erros com média não-zero')
print(f'  MU_ERROR = {MU_ERROR}, GAMMA = {GAMMA}')
: 
,
: { 
: 
 },
: [
]
: 
,
: { 
: 
 },
: [
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,

1

1



1
1

1
1
1
1
,
,
,
2
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,

{'='*60}
,
Simulação para n = {n}
,
{'='*60}
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
